# Flag Smoke Test
Minimal run (2 epochs, 512 train samples) to verify `--bbox_norm` and `--warmup_epochs` flags work end-to-end.

**Docker**: run `bash setup.sh` first, then copy `.env.example` → `.env` and set your paths.  
**Colab**: no `.env` needed — defaults kick in automatically.

In [ ]:
# ── Cell 1: Load environment ──────────────────────────────────────────────────
# On Docker: reads /workspace/Yolo-ST-GCN/.env (set by Duy after cp .env.example .env)
# On Colab:  .env absent → load_dotenv() is a no-op → os.getenv falls back to /content/... defaults

import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv

# Look for .env next to the notebook, then at repo root one level up
_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
for _candidate in [_nb_dir / '.env', _nb_dir.parent / '.env', Path('.env')]:
    if _candidate.exists():
        load_dotenv(_candidate)
        print(f'Loaded .env from {_candidate}')
        break
else:
    print('No .env found — using built-in Colab defaults')

REPO_DIR   = os.getenv('REPO_DIR',   '/content/Yolo-ST-GCN')
BRANCH     = os.getenv('BRANCH',     'refactor-1')
GYM288_PKL = os.getenv('GYM288_PKL', '/content/Gym288-skeleton/gym_288_skeleton.pkl')
GYM99_PKL  = os.getenv('GYM99_PKL',  '/content/Gym99-from-Gym288/gym99_from_gym288.pkl')
OUT_DIR    = os.getenv('OUT_DIR',    'outputs/flag_smoke_test')

print(f'REPO_DIR   = {REPO_DIR}')
print(f'BRANCH     = {BRANCH}')
print(f'GYM288_PKL = {GYM288_PKL}')
print(f'GYM99_PKL  = {GYM99_PKL}')
print(f'OUT_DIR    = {OUT_DIR}')

In [ ]:
# ── Cell 2: Repo setup ────────────────────────────────────────────────────────
# Docker: repo already cloned by setup.sh → just pull latest
# Colab:  repo not present → clone it

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Dataset ───────────────────────────────────────────────────────────
# Docker: GYM288_PKL already on disk → skip download
# Colab:  download from HuggingFace

if Path(GYM288_PKL).exists():
    print(f'Gym288 pickle found: {GYM288_PKL}')
else:
    print('Gym288 pickle not found — downloading from HuggingFace...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'], check=True)
    from huggingface_hub import snapshot_download
    download_dir = Path(GYM288_PKL).parent
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
    pkl_candidates = sorted(download_dir.rglob('*.pkl'))
    if not pkl_candidates:
        raise FileNotFoundError('No .pkl found after Gym288 download.')
    GYM288_PKL = str(pkl_candidates[0])
    print(f'Downloaded: {GYM288_PKL}')

In [ ]:
# ── Cell 4: Smoke train ───────────────────────────────────────────────────────
# 2 epochs so warmup_epochs=1 actually fires the LinearLR→CosineAnnealingLR transition.
# 512 train / 128 val samples — just enough to touch all code paths.

cmd = [
    'python', 'scripts/train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             OUT_DIR,
    '--epochs',              '2',
    '--batch_size',          '64',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--use_two_stream',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--max_train_samples',   '512',
    '--max_val_samples',     '128',
    '--bbox_norm',
    '--warmup_epochs',       '1',
    '--train_data_mode',     'preload_vram',
]

print('Command:')
print(' '.join(cmd))
print()

result = subprocess.run(cmd, check=True)
print('\nSmoke run finished with exit code', result.returncode)

In [ ]:
# ── Cell 5: Verify output ─────────────────────────────────────────────────────
import json

metrics_path = Path(OUT_DIR) / 'metrics_train_gym99.json'
history_path = Path(OUT_DIR) / 'history.json'

print('=== metrics ===')
with open(metrics_path) as f:
    print(json.dumps(json.load(f), indent=2))

print('\n=== history (last epoch) ===')
with open(history_path) as f:
    h = json.load(f)
for k, v in h.items():
    print(f'  {k}: {v}')

print('\nAll output files:')
for p in sorted(Path(OUT_DIR).iterdir()):
    print(' ', p.name)